# Filters: the page as numbers

A scanned page is a grid of numbers. A filter is a small grid of numbers that you lay over every
spot of the page, multiply, and add up. In this notebook you apply filters to a page of a letter
that George Stephens wrote in Copenhagen to his son Joseph.

**Keep your work.** In Colab, choose *File → Save a copy in Drive* first: changes to the copy
opened from GitHub are not saved.

**How to work.** Run one cell at a time with Shift+Enter (or *Runtime → Run all* once, then read
from the top).

In [ ]:
#@title The pages, packed as text. Run this cell; there is no need to open it.
PACKED = "..."


In [ ]:
# The pages travel inside this notebook: the cell above holds a zip file, written out as text.
# This cell turns the text back into files, in a folder next to the notebook.
import base64, pathlib, zipfile
zip_bytes = base64.b64decode(PACKED)
pathlib.Path("workshop-pages.zip").write_bytes(zip_bytes)
zipfile.ZipFile("workshop-pages.zip").extractall("workshop-pages")
ROOT = pathlib.Path("workshop-pages")
PAGES = ROOT / "pages"
print("the page is there:", (PAGES / "letter-stephens-alvin520500-p3.jpg").exists())


In [ ]:
import math
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from scipy.ndimage import correlate

%config InlineBackend.figure_format = "retina"   # sharp pictures on high-resolution screens

def show(images, titles=(), cols=6, width=3):
    """Grey images in a grid of up to cols in a row; 0 is black, 255 is white."""
    cols = min(cols, len(images))
    rows = math.ceil(len(images) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(width * cols, width * 1.35 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for i in range(len(images)):
        axes.flat[i].imshow(images[i], cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        if i < len(titles):
            axes.flat[i].set_title(titles[i], fontsize=9)
    plt.show()

def apply(grid, image):
    """Lay the grid of numbers over every pixel of the image, multiply, add up; divide by the sum of the grid's numbers unless they add up to 0."""
    grid = np.asarray(grid, dtype=float)
    if grid.sum() != 0:
        grid = grid / grid.sum()
    return correlate(image.astype(float), grid, mode="nearest")

## 1. The page as numbers

In [ ]:
page = np.asarray(Image.open(PAGES / "letter-stephens-alvin520500-p3.jpg"))
rows, columns = page.shape
print(rows, "rows,", columns, "columns; smallest value", page.min(), "largest", page.max())
show([page], titles=["George Stephens to his son Joseph, page 3"], width=8)

In [ ]:
# A window of 12 × 12 pixels on the edge of the b in "but": 0 is black ink, 255 is white paper.
top, left = 190, 114
window = page[top:top + 12, left:left + 12]
print(window)
show([window], titles=["the window: one square per number"], width=4)

## 2. A filter by hand

The blur from the lecture lays a 3 × 3 grid of numbers over a pixel and its eight neighbours,
multiplies the nine pairs of numbers, adds them up, and divides by 16, the sum of the grid's
numbers, so that the page keeps its brightness. The two loops below do this for the 12 × 12
window, leaving out the outer rows and columns, where the grid would stick out over the edge.

In [ ]:
# The blur from the lecture: the centre counts most, the corners least.
#
#   1 2 1
#   2 4 2   ÷ 16
#   1 2 1
#
blur_filter = np.array([[1, 2, 1],
                        [2, 4, 2],
                        [1, 2, 1]])
blurred_by_hand = np.zeros((10, 10))
for i in range(10):
    for j in range(10):
        patch = window[i:i + 3, j:j + 3]
        blurred_by_hand[i, j] = (patch * blur_filter).sum() / 16
print(blurred_by_hand.round().astype(int))   # rounded to whole numbers
show([window[1:11, 1:11], blurred_by_hand], titles=["part of the original image", "blur applied 'by hand' (for-loops)"], width=4)

`apply` does nothing but these loops, for every pixel of the page. From here on it does them for
you.

## 3. The filters from the lecture

Each filter is one cell: what it does, the grid of numbers drawn out, the same grid as code. The
pictures show a crop of the page, enlarged, so that the effect can be seen. `apply`
automatically divides by the sum of the grid's numbers, as the lecture divided the blur by 16.

In [ ]:
# A crop of the page, 300 rows by 210 columns.
crop = page[140:440, 20:230]
show([crop], titles=["the crop"], width=4.5)

In [ ]:
# The blur from section 2, on the crop: every pixel becomes a weighted mean of itself and its
# eight neighbours.
show([crop, apply(blur_filter, crop)], titles=["original", "blur 3 × 3"], width=4.5)

In [ ]:
# Vertical edges: minus on the left, plus on the right. Light on the left and dark on the right,
# or the other way round, gives a large number; an even patch gives 0.
#
#   -1  0  1
#   -2  0  2
#   -1  0  1
#
edge_v = np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]])
edges = apply(edge_v, crop)
# The numbers add up to 0, so apply does not divide: the sums run to about a thousand, negative or
# positive. The picture shows their size, ÷ 4 and taken from white: the stronger, the darker.
show([crop, 255 - abs(edges) / 4], titles=["original", "vertical edges"], width=4.5)

In [ ]:
# Horizontal edges: the same grid, turned: minus on top, plus at the bottom. Light above and dark
# below, or the other way round, gives a large number.
#
#   -1 -2 -1
#    0  0  0
#    1  2  1
#
edge_h = np.array([[-1, -2, -1],
                   [ 0,  0,  0],
                   [ 1,  2,  1]])
edges = apply(edge_h, crop)
show([crop, 255 - abs(edges) / 4], titles=["original", "horizontal edges"], width=4.5)

## 4. Your own filter

The cell below starts from the filter that changes nothing: change the numbers, run the cell,
look, and try again. `apply` divides by the sum of your numbers, as by 16 for the blur, so the
page keeps its brightness whatever you type; if they add up to 0, the picture shows the size of
each answer, as for the edges.

In [ ]:
# Nine numbers of your own. As they stand, each pixel counts once and its neighbours not at all.
mine = np.array([[0, 0, 0],
                 [0, 1, 0],
                 [0, 0, 0]])
result = apply(mine, crop)
if mine.sum() == 0:
    result = 255 - abs(result) / 4   # numbers that add up to 0: show the size of each answer
show([crop, result], titles=["original", "result of your filter"], width=4.5)

## 5. A filter for a task: count the lines

So far you have worked with a crop of the page. The task now: how many lines of writing are in
this crop? Nine, as you can count for yourself; but how could a program find out?

If the crop were so blurred that only dark bands on a light background were left, one line of
writing each, there would be a simple way: go down eight columns of the picture, 20 pixels
apart, and count how often each changes from light to dark. When every column gives the same
count, it is very likely the number of lines. The function `dark_stretches` below does this; you
do not need to understand all its details.

The more interesting question is how to get such a blurred picture. Try changing the filter
below, and how often it is applied.

In [ ]:
def dark_stretches(image):
    """Go down eight columns of the image, 20 pixels apart, and count how often each steps from light into dark."""
    counts = np.zeros(8, dtype=int)
    for i in range(8):
        column = image[:, 40 + 20 * i]                        # one column of the picture, top to bottom
        dark = column < column.mean()                         # True where the column is darker than its average
        counts[i] = (np.diff(dark.astype(int)) == 1).sum()   # how often it steps from light into dark
    return counts

print("dark stretches, column by column:", dark_stretches(crop))

In [ ]:
# Your turn: change the filter and how often it is applied, until every column counts the same:
# the number of lines of writing in the crop.
# Hint: read the picture. Thin stripes, several to a line, or lines running into each
# other: the filter smears too little or too much up and down. Gaps between the words that stay
# open: it does not reach far enough sideways yet; each pass smears only about one pixel further.
smear_filter = np.array([[1, 2, 1],
                         [2, 4, 2],
                         [1, 2, 1]])
times = 10   # how often the filter is applied
result = crop
for i in range(times):
    result = apply(smear_filter, result)   # each pass smears the result of the pass before
print("dark stretches, column by column:", dark_stretches(result))
show([crop, result], titles=["original", f"times = {times}"], width=4.5)

## 6. What this was about

Every filter here is a handful of numbers that a person chose: in the lecture, or you. A
convolutional network learns those numbers instead, thousands of them, from examples. Notebook
02 runs the loop that learns such numbers; notebook 03 trains a network of them.